# 🔍 ERCE 2019 · Cuba — Exploración y preprocesamiento del Módulo Nacional (Matemática)

**3er grado · `CUBA3_MAT_MODULO_NACIONAL.csv`**

Este cuaderno es la puerta de entrada al bloque de construcción de modelos. Hace
tres cosas, en este orden:

1. **Explorar** el archivo que entregó el cliente: qué contiene, con qué grano,
   qué está duplicado y qué variables no son lo que su nombre sugiere.
2. **Preprocesar**: una función de curación con un log de decisiones auditable
   que deja el dataset listo.
3. **Seleccionar características** con la misma metodología del proyecto
   (`modelos/seleccion_atributos.py`: CFS, χ², InfoGain y Wrapper, evaluadores de
   Weka), evaluada dentro de una CV para no filtrar información.

> **Hallazgo que gobierna todo el diseño:** los 6 índices de calidad del archivo
> son **medias exactas de sus propios ítems**, y las tres fuentes del módulo
> (estudiante, docente, directivo) son **ortogonales entre sí**. Se demuestra en
> §6 y §7, y de ahí se derivan la regla de exclusión de variables y la
> expectativa sobre los modelos (`cuadernos/estimacion.ipynb`).

## 0. Configuración del entorno

Se carga el CSV con `erce.config.read_csv`, que prueba varias codificaciones: el
archivo **no es UTF-8** (falla con el primer acento de `Saíz`), es `latin-1`.

Del bloque `modelos/` se usan solo las piezas de metodología que el proyecto ya
validó: el preprocesador (`ColumnTransformer`), los cuatro evaluadores de
atributos y la regla de balanceo por índice de desbalance (IR). El cuaderno
`cuadernos/estimacion.ipynb` es el que construye los modelos.

In [1]:
import os
import unicodedata
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# El cuaderno vive en cuadernos/: se sube hasta la raíz del proyecto (donde
# están `erce/` y `modelos/`) para que el flujo works desde cualquier cwd.
RAIZ = Path.cwd().resolve()
for _ in range(3):
    if (RAIZ / "modelos").is_dir():
        break
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

from erce import config as cfg
from modelos import config as mcfg
from modelos import balanceo as BAL
from modelos import preprocesamiento as MPP
from modelos import seleccion_atributos as SA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, StratifiedKFold

RAIZ = mcfg.BASE_DIR
DICCIONARIO = os.path.join(RAIZ, "2. Libros de código")

_candidatas = [cfg.ARCHIVOS["mod_mat"], "CUBA3_MAT_MODULO_NACIONAL.csv"]
RUTA = next((r for r in _candidatas if os.path.exists(r)), _candidatas[0])

raw = cfg.read_csv(RUTA)

print("pandas", pd.__version__)
print("Módulo de modelos:", mcfg.SEED, "(semilla)")
print("Archivo:", os.path.relpath(RUTA, RAIZ))
print("Forma:", raw.shape)

pandas 3.0.5
Módulo de modelos: 42 (semilla)
Archivo: 07.ERCE-2019-FINAL/ICCP-BBDD FINALES/CUBA3_MAT_MODULO_NACIONAL.csv
Forma: (5595, 49)


## 1. Estructura del archivo

5.595 filas × 49 columnas. Las 49 columnas caen en familias muy limpias: tres
bloques de ítems (respondidos por el estudiante, el docente y el directivo), seis
índices derivados, contexto de la escuela y una capa de identificadores.

In [2]:
FIGURA_ARCHIVO = {
    "Identificadores": ["IDSCHOOL", "IDCLASS", "IDSTUD", "Id_prof", "school"],
    "Escuela y territorio": ["nomb_esc", "geo_esc_x", "geo_esc_y", "ubi1_cen_esc",
                             "cod1_ubi_cen", "ubi2_cen_esc", "ubi3_cen_esc",
                             "num_hab_esc", "area_esc"],
    "Contexto del estudiante": ["nomb_est", "test", "edad", "género",
                                "sec_3", "can_prof_3", "prof_hom_3", "prof_muj_3"],
    "Índices de calidad": ["Indice_preg", "Indice_refuerzo", "Indice_global",
                           "Indice_prep_didact_prof", "Indice_calidad_clases_prof",
                           "Indice_calidad_clases_dir"],
    "Ítems del estudiante (E3IT21)": [c for c in raw.columns if c.startswith("E3IT21")],
    "Ítems del docente (PPIT)": [c for c in raw.columns if c.startswith("PPIT")],
    "Ítems del directivo (DDIT41)": [c for c in raw.columns if c.startswith("DDIT41")],
}

familias = pd.DataFrame(
    [{"familia": k, "n_columnas": len(v), "columnas": ", ".join(v)}
     for k, v in FIGURA_ARCHIVO.items()])

print(f"Memoria en memoria: {raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nTipos de dato:\n", raw.dtypes.value_counts().to_frame("columnas").T.to_string())
familias

Memoria en memoria: 2.65 MB

Tipos de dato:
           int64  str  float64
columnas     39    8        2


,familia,n_columnas,columnas
0,Identificadores,5,"IDSCHOOL, IDCLASS, IDSTUD, Id_prof, school"
1,Escuela y territorio,9,"nomb_esc, geo_esc_x, geo_esc_y, ubi1_cen_esc, ..."
2,Contexto del estudiante,8,"nomb_est, test, edad, género, sec_3, can_prof_..."
3,Índices de calidad,6,"Indice_preg, Indice_refuerzo, Indice_global, I..."
4,Ítems del estudiante (E3IT21),5,"E3IT21_01_N, E3IT21_02_N, E3IT21_03_N, E3IT21_..."
5,Ítems del docente (PPIT),11,"PPIT44_01_N, PPIT44_02_N, PPIT44_03_N, PPIT44_..."
6,Ítems del directivo (DDIT41),5,"DDIT41_01_N, DDIT41_02_N, DDIT41_03_N, DDIT41_..."


In [3]:
# Qué NO trae el archivo (relevante para la elección del objetivo)
NO_TRAE = {
    "Puntaje de Matemática (VP1..VP5)": [c for c in raw.columns if c.startswith("VP")],
    "Nivel de desempeño (NL_VP1)": [c for c in raw.columns if c.startswith("NL_")],
    "Pesos de muestreo (WT/WSEN/WS)": [c for c in raw.columns if c in ("WT", "WSEN", "WS")],
    "Índice socioeconómico (ISECF)": [c for c in raw.columns if c.upper() == "ISECF"],
}

pd.DataFrame(
    [{"variable": k, "columnas encontradas": ", ".join(v) if v else "— ninguna —",
      "consecuencia": "no se puede usar sin unir otra base" if not v else "presente"}
     for k, v in NO_TRAE.items()])

,variable,columnas encontradas,consecuencia
0,Puntaje de Matemática (VP1..VP5),— ninguna —,no se puede usar sin unir otra base
1,Nivel de desempeño (NL_VP1),— ninguna —,no se puede usar sin unir otra base
2,Pesos de muestreo (WT/WSEN/WS),— ninguna —,no se puede usar sin unir otra base
3,Índice socioeconómico (ISECF),— ninguna —,no se puede usar sin unir otra base


## 2. Claves e integridad

Los identificadores son jerárquicos y sus dígitos lo confirman: escuela (4) +
sección (2) + alumno (2), y el docente va en los 4 primeros dígitos de `Id_prof`.
Se valida con aritmética, no con suposiciones.

Dos trampas de clave: `nomb_esc` **no es única** (16 nombres de escuela
corresponden a más de un `IDSCHOOL`) y `school` es un código distinto del
`IDSCHOOL` interno de LLECE.

In [4]:
assert (raw["IDCLASS"] // 100 == raw["IDSCHOOL"]).all()
assert (raw["IDSTUD"] // 10000 == raw["IDSCHOOL"]).all()
assert ((raw["IDSTUD"] % 10000) // 100 == raw["IDCLASS"] % 100).all()
assert (raw["Id_prof"] // 10000 == raw["IDSCHOOL"]).all()
assert (raw["test"] == "M3").all()

claves = pd.DataFrame([
    {"clave": "IDSCHOOL", "únicos": raw.IDSCHOOL.nunique(), "grano": "escuela"},
    {"clave": "IDCLASS", "únicos": raw.IDCLASS.nunique(), "grano": "sección"},
    {"clave": "IDSTUD", "únicos": raw.IDSTUD.nunique(), "grano": "estudiante"},
    {"clave": "Id_prof", "únicos": raw["Id_prof"].nunique(), "grano": "docente"},
    {"clave": "school", "únicos": raw.school.nunique(), "grano": "código externo de escuela"},
    {"clave": "nomb_esc", "únicos": raw.nomb_esc.nunique(), "grano": "nombre (¡no es clave!)"},
])

por_clase = raw.drop_duplicates("IDSTUD").groupby("IDCLASS").size()
print("Jerarquía verificada con aritmética: IDCLASS//100 = IDSCHOOL, "
      "IDSTUD//10000 = IDSCHOOL, Id_prof//10000 = IDSCHOOL")
print("Secciones por escuela (máximo):", raw.groupby("IDSCHOOL")["IDCLASS"].nunique().max())
print("\nEstudiantes por sección:\n", por_clase.describe().round(1).to_string())
claves

Jerarquía verificada con aritmética: IDCLASS//100 = IDSCHOOL, IDSTUD//10000 = IDSCHOOL, Id_prof//10000 = IDSCHOOL
Secciones por escuela (máximo): 1

Estudiantes por sección:
 count   228.000
mean     21.500
std       8.900
min       1.000
25%      17.000
50%      22.000
75%      27.000
max      52.000


,clave,únicos,grano
0,IDSCHOOL,228,escuela
1,IDCLASS,228,sección
2,IDSTUD,4899,estudiante
3,Id_prof,252,docente
4,school,228,código externo de escuela
5,nomb_esc,204,nombre (¡no es clave!)


In [5]:
# El nombre de la escuela no identifica una escuela
compartidas = (raw.groupby("nomb_esc")["IDSCHOOL"].nunique()
               .sort_values(ascending=False))
print(f"nombres de escuela que corresponden a más de un IDSCHOOL: {(compartidas > 1).sum()} "
      f"de {len(compartidas)}")
ejemplo = compartidas.index[0]
(raw[raw.nomb_esc == ejemplo]
 .groupby(["IDSCHOOL", "school", "ubi1_cen_esc", "ubi2_cen_esc"])
 .size().rename("estudiantes").reset_index())

nombres de escuela que corresponden a más de un IDSCHOOL: 16 de 204


,IDSCHOOL,school,ubi1_cen_esc,ubi2_cen_esc,estudiantes
0,1051,21092302,Pinar del Río,San Luis,12
1,1108,23090041,La Habana,Diez de Octubre,18
2,1140,25067105,Matanzas,Jovellanos,26
3,1178,28022158,Sancti Spíritus,Jatibonico,20
4,1219,31022445,Las Tunas,Puerto Padre,19


## 3. El grano real: 5.595 filas, 4.899 estudiantes

696 estudiantes aparecen **dos veces**. La causa no es la misma para todos, y
conviene separarla:

- **678** tienen dos docentes distintos: el módulo lo respondió cada docente de
  la sección, y sus respuestas a ciertos ítems no siempre coinciden.
- **18** tienen el mismo `Id_prof`: son los 18 estudiantes de la escuela 1096, con
  la ficha de la escuela repetida y datos de localidad distintos.

Después se comparan las tres estrategias para resolver los duplicados, con el sesgo
que cada una introduce.

In [6]:
dup = raw["IDSTUD"].duplicated(keep=False)
dd = raw[dup].sort_values(["IDSTUD", "Id_prof"])
pos = dd.groupby("IDSTUD").cumcount()

n_docs = dd.groupby("IDSTUD")["Id_prof"].nunique()
print(f"estudiantes duplicados: {dd.IDSTUD.nunique()}")
print(f"  · con 2 docentes distintos : {(n_docs == 2).sum()}")
print(f"  · con el mismo Id_prof      : {(n_docs == 1).sum()} "
      f"(escuela(s) {[int(s) for s in sorted(dd[dd.IDSTUD.isin(n_docs[n_docs == 1].index)].IDSCHOOL.unique())]})")

filas = []
for col in raw.columns:
    serie = pd.to_numeric(dd[col], errors="coerce")
    if serie.notna().all():
        izq = serie[pos == 0].to_numpy()
        der = serie[pos == 1].to_numpy()
        cambiados = int((np.abs(izq - der) > 0).sum())
        delta = float(np.abs(izq - der).mean())
    else:
        a = dd[col][pos == 0].astype(str).str.strip().str.lower().to_numpy()
        b = dd[col][pos == 1].astype(str).str.strip().str.lower().to_numpy()
        cambiados = int((a != b).sum())
        delta = np.nan
    if cambiados:
        filas.append({"columna": col, "estudiantes afectados": cambiados,
                      "% de los duplicados": round(100 * cambiados / len(n_docs), 1),
                      "|Δ| medio entre las 2 filas": delta})

diferencias = (pd.DataFrame(filas)
               .sort_values("estudiantes afectados", ascending=False)
               .reset_index(drop=True))
print("\nColumnas que NO son constantes dentro de un mismo estudiante:")
diferencias

estudiantes duplicados: 696
  · con 2 docentes distintos : 678
  · con el mismo Id_prof      : 18 (escuela(s) [1096])

Columnas que NO son constantes dentro de un mismo estudiante:


,columna,estudiantes afectados,% de los duplicados,|Δ| medio entre las 2 filas
0,Id_prof,678,97.400,98.388
1,PPIT44_02_N,118,17.000,0.170
2,Indice_prep_didact_prof,113,16.200,0.101
3,PPIT45_N,110,15.800,0.158
4,PPIT44_05_N,93,13.400,0.134
5,PPIT44_01_N,86,12.400,0.124
6,Indice_calidad_clases_prof,75,10.800,0.022
7,PPIT44_04_N,61,8.800,0.088
8,PPIT44_03_N,52,7.500,0.075
9,PPIT46_02_N,39,5.600,0.056


In [7]:
# Tres estrategias para resolver los duplicados
ITEM_PROF = [c for c in raw.columns if c.startswith("PPIT")]
IDX_PROF = ["Indice_prep_didact_prof", "Indice_calidad_clases_prof"]
VARIANTES = ITEM_PROF + IDX_PROF

# A: exactamente lo que hace erce/data.py -> conserva al docente 01
dedup = dd.sort_values("Id_prof").drop_duplicates("IDSTUD", keep="first")
A = dedup.set_index("IDSTUD")[VARIANTES]
# B: promedio de los docentes que respondieron
B = dd.groupby("IDSTUD")[VARIANTES].mean()
# C: promedio de la escuela del estudiante
media_escuela = raw.groupby("IDSCHOOL")[IDX_PROF].mean()
escuela_de = dedup.set_index("IDSTUD")["IDSCHOOL"]
C = pd.DataFrame({c: escuela_de.map(media_escuela[c]) for c in IDX_PROF}, index=A.index)

comparacion = pd.DataFrame(index=VARIANTES)
comparacion["A · 1ª fila (erce/data.py)"] = A.mean()
comparacion["B · promedio de docentes"] = B.mean()
comparacion["A ≠ B"] = [(A[c] - B[c]).abs().gt(0).sum() for c in VARIANTES]
comparacion["max |A − B|"] = (A - B).abs().max()

print("Estrategia A conserva siempre al docente 01:",
      int(A.index.to_series().map(dedup.set_index("IDSTUD")["Id_prof"] % 100).eq(1).sum()),
      "de", len(A), "estudiantes")
print("A y B coinciden en", int(((A - B).abs().sum(axis=1) == 0).sum()), "de", len(A), "estudiantes")
print("Un estudiante nunca cambia de escuela entre sus filas:",
      bool(dd.groupby("IDSTUD")["IDSCHOOL"].nunique().le(1).all()))
print("C (media de la escuela) vs B, índice de preparación docente: "
      f"{float((C['Indice_prep_didact_prof'] - B['Indice_prep_didact_prof']).abs().max()):.4f}")
comparacion.round(4)

Estrategia A conserva siempre al docente 01: 696 de 696 estudiantes
A y B coinciden en 440 de 696 estudiantes
Un estudiante nunca cambia de escuela entre sus filas: True
C (media de la escuela) vs B, índice de preparación docente: 0.0000


,A · 1ª fila (erce/data.py),B · promedio de docentes,A ≠ B,max |A − B|
PPIT44_01_N,0.534,0.473,86,0.500
PPIT44_02_N,0.428,0.382,118,0.500
PPIT44_03_N,0.428,0.430,52,0.500
PPIT44_04_N,0.428,0.384,61,0.500
PPIT44_05_N,0.447,0.462,93,0.500
PPIT45_N,0.770,0.786,110,0.500
PPIT46_01_N,1.000,1.000,0,0.000
PPIT46_02_N,0.950,0.922,39,0.500
PPIT46_03_N,0.951,0.925,36,0.500
PPIT46_04_N,1.000,1.000,0,0.000


**Qué se recomienda y por qué**

Se adopta la **estrategia B (promediar lo que responde el docente)** para el
análisis a nivel de estudiante:

- **A** (lo que hace hoy `erce/data.py`: `sort_values("Id_prof")` + `drop_duplicates`)
  no es neutra: como el menor `Id_prof` es siempre el del docente 01, conserva su
  respuesta en los 4.899 estudiantes y descarta la del docente 02 en los 256
  estudiantes (37 % de los 696 duplicados) donde el bloque docente difiere; la
  desviación máxima llega a 0,4 en el índice de preparación docente. Es un sesgo
  de registro, no una decisión de análisis.
- **B** usa los dos informes disponibles y no pierde al estudiante; introduce una
  media en variables 0/1 (fracción de acuerdo entre los dos docentes), lo que es
  coherente con cómo el propio módulo construye sus índices.
- **C** (media de la escuela) da aquí exactamente lo mismo que B (diferencia
  máxima 0,00), porque todos los estudiantes de una sección comparten el mismo
  par de docentes. Es la correcta si el objeto de estudio es la escuela, pero
  colapsa la variación entre estudiantes de una misma sección.

Los ítems del estudiante (`E3IT21`), los índices del estudiante y el contexto
escolar **no** cambian entre las dos filas: la duplicación es enteramente del lado
del docente.

## 4. Calidad de los datos

Sin nulos en las 5.595×49 celdas, pero con cuatro defectos que hay que corregir
antes de modelar, y uno que hay que entender:

| Defecto | Qué pasa | Dónde |
|---|---|---|
| Coordenadas como texto | `geo_esc_x`/`geo_esc_y` llegan como `str` | columnas |
| Localidad escrita de dos maneras | `San Agustín` / `San Agustin` | 18 estudiantes, escuela 1096 |
| Columnas redundantes | `sec_3` = `can_prof_3`, y `can_prof_3` = `prof_hom_3 + prof_muj_3` | 3 columnas |
| Escalas mezcladas | dos índices en 0–1 y cuatro en 0–100 | índices |

In [8]:
faltantes = raw.isna().sum()
print("celdas nulas:", int(faltantes.sum()),
      "| columnas con nulos:", int((faltantes > 0).sum()))

print("\nCoordenadas como texto:")
for c in ("geo_esc_x", "geo_esc_y"):
    print(f"  {c}: dtype={raw[c].dtype}  ejemplos={raw[c].head(3).tolist()}")

geo = raw.assign(lat=pd.to_numeric(raw.geo_esc_x, errors="coerce"),
                 lon=pd.to_numeric(raw.geo_esc_y, errors="coerce"))
print(f"\nrango: lat {geo.lat.min():.4f}..{geo.lat.max():.4f} | "
      f"lon {geo.lon.min():.4f}..{geo.lon.max():.4f}  (Cuba ≈ lat 19.8..23.3 / lon -84.9..-74.1)")


def sin_acentos(texto):
    return "".join(c for c in unicodedata.normalize("NFD", str(texto))
                   if unicodedata.category(c) != "Mn")


variantes = (raw.assign(ubi3_norm=raw.ubi3_cen_esc.map(sin_acentos).str.strip())
                 .query("ubi3_cen_esc != ubi3_norm")[["IDSCHOOL", "ubi3_cen_esc"]]
                 .drop_duplicates())
print("\nTextos que cambian al normalizar acentos/espacios:")
variantes

incoherentes = (raw.groupby("IDSCHOOL")[["geo_esc_x", "geo_esc_y", "ubi3_cen_esc"]]
                .nunique())
print("\nEscuelas con más de un valor por campo geográfico:")
incoherentes[(incoherentes > 1).any(axis=1)]

celdas nulas: 0 | columnas con nulos: 0

Coordenadas como texto:
  geo_esc_x: dtype=str  ejemplos=['22.4143358', '22.4143358', '22.4143358']
  geo_esc_y: dtype=str  ejemplos=['-83.626369', '-83.626369', '-83.626369']

rango: lat 20.1602..23.8732 | lon -84.3424..-75.5394  (Cuba ≈ lat 19.8..23.3 / lon -84.9..-74.1)

Textos que cambian al normalizar acentos/espacios:

Escuelas con más de un valor por campo geográfico:


,geo_esc_x,geo_esc_y,ubi3_cen_esc
IDSCHOOL,,,
1096,2,2,2


In [9]:
print("Redundancias exactas:")
print("  sec_3 == can_prof_3                    :", bool((raw.sec_3 == raw.can_prof_3).all()))
print("  can_prof_3 == prof_hom_3 + prof_muj_3 :",
      bool((raw.can_prof_3 == raw.prof_hom_3 + raw.prof_muj_3).all()))
print("  test (constante)                       :", raw.test.unique().tolist())

print("\nEscala de cada índice (el módulo no usa una sola):")
filas_indice = []
for c in FIGURA_ARCHIVO["Índices de calidad"]:
    v = sorted(raw[c].unique())
    filas_indice.append({
        "índice": c, "n valores": raw[c].nunique(), "valores": str(v[:6]) + ("…" if len(v) > 6 else ""),
        "escala": "0–1" if raw[c].max() <= 1 else "0–100",
        "valor modal": raw[c].mode().iat[0],
        "% en el valor modal": round(100 * raw[c].value_counts(normalize=True).max(), 1),
    })
pd.DataFrame(filas_indice)

Redundancias exactas:
  sec_3 == can_prof_3                    : True
  can_prof_3 == prof_hom_3 + prof_muj_3 : True
  test (constante)                       : ['M3']

Escala de cada índice (el módulo no usa una sola):


,índice,n valores,valores,escala,valor modal,% en el valor modal
0,Indice_preg,3,"[np.int64(0), np.int64(50), np.int64(100)]",0–100,50.000,35.200
1,Indice_refuerzo,3,"[np.int64(0), np.int64(50), np.int64(100)]",0–100,100.000,35.400
2,Indice_global,6,"[np.int64(0), np.int64(20), np.int64(40), np.i...",0–100,0.000,21.600
3,Indice_prep_didact_prof,6,"[np.float64(0.0), np.float64(0.2), np.float64(...",0–1,1.000,39.800
4,Indice_calidad_clases_prof,6,"[np.float64(0.0), np.float64(0.2), np.float64(...",0–1,1.000,88.100
5,Indice_calidad_clases_dir,6,"[np.int64(0), np.int64(20), np.int64(40), np.i...",0–100,100.000,86.500


### 4.1 `edad` no es la edad

`edad` toma los valores 1 a 8 y se concentra en 2 y 3. Ningún estudiante de 3er
grado tiene 2 años: es un **código de banda**. La decodificación exacta
(`edad_anos = edad + 6`, es decir de 7 a 14 años) se comprueba en la celda siguiente
contra la base ERCE, que es el único sitio donde la edad está en años.

In [10]:
print("Distribución de `edad` tal como viene:")
raw.edad.value_counts().sort_index().rename("estudiantes").to_frame()

Distribución de `edad` tal como viene:


,estudiantes
edad,
1,30
2,3336
3,2132
4,57
5,8
6,29
7,2
8,1


In [11]:
# Comprobación externa (fuera de este CSV): la base ERCE guarda EDAD en años.
_fa = cfg.ARCHIVOS["fa"]
if os.path.exists(_fa):
    fa = cfg.read_csv(_fa)[["IDSTUD", "EDAD"]].drop_duplicates("IDSTUD")
    cruce = raw.drop_duplicates("IDSTUD")[["IDSTUD", "edad"]].merge(fa, on="IDSTUD")
    print(pd.crosstab(cruce.edad, cruce.EDAD))
    con_edad = cruce[cruce["EDAD"].notna()]
    assert (con_edad["EDAD"] == con_edad["edad"] + 6).all()
    print(f"\nVerificado: EDAD = edad + 6 en los {len(con_edad)} estudiantes emparejados; "
          f"{int(cruce['EDAD'].isna().sum())} no tienen EDAD en FA_A3 y quedan fuera.")
else:
    print("Base ERCE no disponible; la decodificación queda como hipótesis documentada.")

EDAD  7.000   8.000   9.000   10.000  11.000  12.000
edad                                                
1         26       0       0       0       0       0
2          0    2876       0       0       0       0
3          0       0    1873       0       0       0
4          0       0       0      53       0       0
5          0       0       0       0       8       0
6          0       0       0       0       0      21

Verificado: EDAD = edad + 6 en los 4857 estudiantes emparejados; 22 no tienen EDAD en FA_A3 y quedan fuera.


## 5. Diccionario de las 49 columnas

Con las marcas de atención: lo que no es lo que parece (`edad`), lo que está
en una escala distinta a la del resto, y lo que apenas varía (porcentaje alto en
un solo valor → poca información y riesgo de sobreajuste).

In [12]:
NOTAS = {
    "edad": "código de banda 1–8, NO años (decodificar: +6)",
    "género": "1 = Niño, 2 = Niña",
    "area_esc": "1 = Urbana, 2 = Rural",
    "geo_esc_x": "latitud como texto", "geo_esc_y": "longitud como texto",
    "cod1_ubi_cen": "código 21–33 de provincia (1-1 con ubi1_cen_esc)",
    "sec_3": "duplicada: sec_3 == can_prof_3",
    "can_prof_3": "duplicada: = prof_hom_3 + prof_muj_3",
    "Indice_prep_didact_prof": "escala 0–1 (el resto de índices, 0–100)",
    "Indice_calidad_clases_prof": "escala 0–1 (el resto de índices, 0–100)",
    "Indice_calidad_clases_dir": "escala 0–100 pero 85 % en el valor 100",
    "Indice_global": "20 × suma de los 5 ítems E3IT21 (ver §6)",
    "Indice_preg": "50 × (E3IT21_01 + E3IT21_04) (ver §6)",
    "Indice_refuerzo": "50 × (E3IT21_02 + E3IT21_03) (ver §6)",
    "test": "constante 'M3'",
    "nomb_est": "nombre del estudiante con dígitos (anonimizado)",
    "nomb_esc": "nombre de la escuela; NO es clave",
    "school": "código externo, distinto de IDSCHOOL",
    "Id_prof": "docente; se repite en 696 estudiantes",
}

diccionario = []
for c in raw.columns:
    s = raw[c]
    if pd.api.types.is_numeric_dtype(s):
        rango = f"{s.min():g} .. {s.max():g}"
        modal = f"{s.mode().iat[0]:g} ({100 * s.value_counts(normalize=True).max():.1f} %)"
    else:
        vals = s.value_counts()
        rango = f"{s.nunique()} categorías"
        modal = f"{vals.index[0]} ({100 * vals.max() / len(s):.1f} %)"
    diccionario.append({
        "columna": c, "tipo": str(s.dtype), "n únicos": s.nunique(),
        "rango": rango, "modal": modal,
        "marca": NOTAS.get(c, ""),
    })

diccionario = pd.DataFrame(diccionario)
print("columnas con marca de atención:", int((diccionario.marca != "").sum()))
diccionario

columnas con marca de atención: 19


,columna,tipo,n únicos,rango,modal,marca
0,IDSCHOOL,int64,228,1001 .. 1249,1153 (1.5 %),
1,IDCLASS,int64,228,100101 .. 124902,115301 (1.5 %),
2,IDSTUD,int64,4899,1.00101e+07 .. 1.24902e+07,1.00402e+07 (0.0 %),
3,nomb_est,str,4899,4899 categorías,Diego José Alavrez Hernánez (0.0 %),nombre del estudiante con dígitos (anonimizado)
4,nomb_esc,str,204,204 categorías,José Antonio Echeverría Bianchi (2.9 %),nombre de la escuela; NO es clave
5,school,int64,228,2.10126e+07 .. 3.31124e+07,2.60901e+07 (1.5 %),"código externo, distinto de IDSCHOOL"
6,geo_esc_x,str,229,229 categorías,22.399611 (1.5 %),latitud como texto
7,geo_esc_y,str,229,229 categorías,-79.96368 (1.5 %),longitud como texto
8,ubi1_cen_esc,str,13,13 categorías,La Habana (21.9 %),
9,cod1_ubi_cen,int64,13,21 .. 33,23 (21.9 %),código 21–33 de provincia (1-1 con ubi1_cen_esc)


## 6. Los seis índices son medias de sus propios ítems

Este es el hallazgo que decide el diseño. Los seis índices no aportan información
adicional: son **la media de los ítems que los originan**, calculada sobre la
misma fila.

| Índice | Fórmula | Coincide |
|---|---|---|
| `Indice_global` | 20 × Σ(`E3IT21_01..05`) | 5.595 / 5.595 |
| `Indice_preg` | 50 × (`E3IT21_01` + `E3IT21_04`) | 5.595 / 5.595 |
| `Indice_refuerzo` | 50 × (`E3IT21_02` + `E3IT21_03`) | 5.595 / 5.595 |
| `Indice_prep_didact_prof` | media(`PPIT44_01..05`) | 5.595 / 5.595 |
| `Indice_calidad_clases_prof` | media(`PPIT46_01..05`) | 5.595 / 5.595 |
| `Indice_calidad_clases_dir` | media(`DDIT41_01..05`) × 100 | 5.595 / 5.595 |

**Por qué importa:** si el objetivo es `Indice_global` y el conjunto de variables
incluye los cinco ítems que lo forman, la regresión no aprende nada: reproduce
una suma. Se demuestra abajo con R² = 1.0000 exacto.

In [13]:
E3 = [f"E3IT21_0{i}_N" for i in range(1, 6)]
PPIT44 = [f"PPIT44_0{i}_N" for i in range(1, 6)]
PPIT45 = ["PPIT45_N"]
PPIT46 = [f"PPIT46_0{i}_N" for i in range(1, 6)]
DDIT41 = [f"DDIT41_0{i}_N" for i in range(1, 6)]

FORMULAS = {
    "Indice_global": ("20 × Σ E3IT21_01..05", lambda d: 20 * d[E3].sum(axis=1)),
    "Indice_preg": ("50 × (E3IT21_01 + E3IT21_04)",
                    lambda d: 50 * (d["E3IT21_01_N"] + d["E3IT21_04_N"])),
    "Indice_refuerzo": ("50 × (E3IT21_02 + E3IT21_03)",
                        lambda d: 50 * (d["E3IT21_02_N"] + d["E3IT21_03_N"])),
    "Indice_prep_didact_prof": ("media PPIT44_01..05", lambda d: d[PPIT44].mean(axis=1)),
    "Indice_calidad_clases_prof": ("media PPIT46_01..05", lambda d: d[PPIT46].mean(axis=1)),
    "Indice_calidad_clases_dir": ("media DDIT41_01..05 × 100",
                                  lambda d: 100 * d[DDIT41].mean(axis=1)),
}

verificacion = []
for nombre, (formula, fn) in FORMULAS.items():
    calculado = fn(raw)
    exacto = int(np.isclose(calculado, raw[nombre]).sum())
    verif_row = {"índice": nombre, "fórmula": formula, "coincide": exacto, "de": len(raw),
                 "correlación": round(float(raw[nombre].corr(calculado)), 6)}
    verificacion.append(verif_row)
    assert exacto == len(raw), f"{nombre} no se reproduce exactamente"

verificacion = pd.DataFrame(verificacion)
print("Las seis fórmulas se reproducen en todas las filas del archivo.\n")
verificacion

Las seis fórmulas se reproducen en todas las filas del archivo.



,índice,fórmula,coincide,de,correlación
0,Indice_global,20 × Σ E3IT21_01..05,5595,5595,1.000
1,Indice_preg,50 × (E3IT21_01 + E3IT21_04),5595,5595,1.000
2,Indice_refuerzo,50 × (E3IT21_02 + E3IT21_03),5595,5595,1.000
3,Indice_prep_didact_prof,media PPIT44_01..05,5595,5595,1.000
4,Indice_calidad_clases_prof,media PPIT46_01..05,5595,5595,1.000
5,Indice_calidad_clases_dir,media DDIT41_01..05 × 100,5595,5595,1.000


In [14]:
# Demostración de la fuga: regresión lineal con los 5 ítems que forman el objetivo
X_items = raw[E3].to_numpy(float)
y_indice = raw["Indice_global"].to_numpy(float)

r2_con_items = r2_score(y_indice, LinearRegression().fit(X_items, y_indice).predict(X_items))
contexto = ["edad", "num_hab_esc", "can_prof_3", "prof_hom_3", "prof_muj_3"]
X_ctx = raw[contexto].to_numpy(float)
r2_con_contexto = r2_score(y_indice, LinearRegression().fit(X_ctx, y_indice).predict(X_ctx))

print(f"R²(Indice_global ~ sus 5 ítems E3IT21)      = {r2_con_items:.6f}   ← fuga")
print(f"R²(Indice_global ~ contexto escolar)        = {r2_con_contexto:.6f}   ← señal real, si la hay")
print("\nRegla que se aplica en §8: los componentes del objetivo nunca entran")
print("como variables del modelo (E3IT21_*, Indice_preg, Indice_refuerzo).")

R²(Indice_global ~ sus 5 ítems E3IT21)      = 1.000000   ← fuga
R²(Indice_global ~ contexto escolar)        = 0.001199   ← señal real, si la hay

Regla que se aplica en §8: los componentes del objetivo nunca entran
como variables del modelo (E3IT21_*, Indice_preg, Indice_refuerzo).


## 7. Las tres fuentes del módulo no se hablan entre sí

Los índices del estudiante correlacionan 0.87–0.88 entre sí (son la misma
dimensión medida dos veces, §6). Pero **entre bloques distintos la correlación
es prácticamente cero**: lo que percibe el estudiante no se parece a lo que
percibe el docente sobre sí mismo ni a lo que percibe el directivo.

El mapa de abajo trabaja con los **ítems crudos** (escala 0–1), no con los
índices: por dentro, el bloque del estudiante es el más coherente (|r| medio
0.420 entre sus cinco ítems) y aun así con cualquier otro bloque no pasa de
0.030; los valores más altos fuera de la diagonal son PPIT44–PPIT46 (0.160),
que son bloques del mismo respondiente.

La correlación con el objetivo se comporta igual, con una excepción que conviene
nombrar: el bloque del estudiante (los `E3IT21`) es el único asociado al objetivo,
y lo es por la definición misma del índice (§6). Excluido ese bloque circular, la
asociación de todas las variables disponibles con el objetivo es ruido.

Esto fija la expectativa del cuaderno `cuadernos/estimacion.ipynb`: con un solo objetivo, las
variables disponibles son casi ortogonales a él, así que lo esperable es un
R² ≈ 0 o un F1 que no supere la línea base. Que eso ocurra **es el resultado**, y
por eso todos los números se reportan contra la línea base de siempre acertar lo
más frecuente.

In [15]:
BLOQUES = {
    "Estudiante · E3IT21": E3,
    "Docente · PPIT44 (preparación)": PPIT44,
    "Docente · PPIT45": PPIT45,
    "Docente · PPIT46 (calidad de sus clases)": PPIT46,
    "Directivo · DDIT41": DDIT41,
    "Contexto escolar": ["edad", "num_hab_esc", "can_prof_3", "prof_hom_3", "prof_muj_3"],
}

# Una sola correlación de todos los bloques, cortada por pares.
# (No usar `corrwith` entre frames: alinea las columnas por nombre y los
# bloques no tienen nombres en común, así que la matriz salía entera a NaN.)
cols_todos = [c for cols in BLOQUES.values() for c in cols]
C = raw[cols_todos].corr()

# |r| medio entre pares de columnas; en la diagonal, dentro del propio bloque
matriz = pd.DataFrame(index=BLOQUES, columns=BLOQUES, dtype=float)
for a in BLOQUES:
    for b in BLOQUES:
        bloque = np.abs(C.loc[BLOQUES[a], BLOQUES[b]].to_numpy())
        if a == b:
            # dentro del bloque, sin la diagonal trivial (los 1,0)
            v = bloque[~np.eye(len(bloque), dtype=bool)] if len(bloque) > 1 else np.array([])
        else:
            v = bloque.ravel()
        matriz.loc[a, b] = v.mean() if v.size else np.nan  # PPIT45: 1 sola columna, sin pares

matriz.index.name = "bloque A"
matriz

,Estudiante · E3IT21,Docente · PPIT44 (preparación),Docente · PPIT45,Docente · PPIT46 (calidad de sus clases),Directivo · DDIT41,Contexto escolar
bloque A,,,,,,
Estudiante · E3IT21,0.420,0.030,0.026,0.029,0.015,0.016
Docente · PPIT44 (preparación),0.030,0.778,0.033,0.160,0.046,0.056
Docente · PPIT45,0.026,0.033,NaN,0.151,0.086,0.074
Docente · PPIT46 (calidad de sus clases),0.029,0.160,0.151,0.533,0.107,0.038
Directivo · DDIT41,0.015,0.046,0.086,0.107,0.256,0.050
Contexto escolar,0.016,0.056,0.074,0.038,0.050,0.169


In [16]:
fig = go.Figure(go.Heatmap(
    z=matriz.values, x=list(matriz.columns), y=list(matriz.index),
    colorscale=[[0, "#1f3a93"], [0.5, "#fbfbfa"], [1, "#c8102e"]],
    zmin=0, zmax=max(0.12, np.nanmax(matriz.values)),
    text=np.round(matriz.values, 3), texttemplate="%{text}",
    colorbar={"title": "|r| medio"}))
fig.update_layout(
    title={"text": "Correlación media |r| entre bloques (diagonal: dentro del bloque)",
           "x": 0.5, "xanchor": "center", "font": {"size": 17}},
    height=430, template="plotly_white", margin={"l": 210, "r": 20, "t": 60, "b": 90},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 12},
    xaxis={"tickangle": -35}, yaxis={"autorange": "reversed"})
fig.show()

In [17]:
vs_objetivo = pd.DataFrame([
    {"bloque": b,
     "|r| medio": round(float(raw[cols].corrwith(raw["Indice_global"]).abs().mean()), 4),
     "|r| máximo": round(float(raw[cols].corrwith(raw["Indice_global"]).abs().max()), 4)}
    for b, cols in BLOQUES.items()]).sort_values("|r| medio")

fig = go.Figure(go.Bar(
    x=vs_objetivo["|r| medio"], y=vs_objetivo.bloque, orientation="h",
    marker_color="#1f3a93",
    text=vs_objetivo["|r| medio"].map("{:.3f}".format), textposition="outside",
    hovertemplate="%{y}<br>|r| medio con Indice_global: %{x:.3f}<extra></extra>"))
fig.update_layout(
    title={"text": "Asociación de cada bloque con el objetivo (Indice_global)",
           "x": 0.5, "xanchor": "center", "font": {"size": 17}},
    height=380, template="plotly_white", margin={"l": 230, "r": 60, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 13},
    xaxis={"title": "|r| medio con Indice_global", "gridcolor": "#e8e8e8"},
    yaxis={"autorange": "reversed"})
fig.show()

print("máximo absoluto en todo el archivo:",
      round(float(vs_objetivo['|r| máximo'].max()), 4),
      "→ es el bloque circular del estudiante, y solo ese")
no_circ = vs_objetivo[~vs_objetivo.bloque.str.startswith("Estudiante")]
print("máximo |r| en los bloques NO circulares:",
      round(float(no_circ['|r| máximo'].max()), 4),
      "→ ninguna variable que vaya a entrar al modelo se asocia con el objetivo")
vs_objetivo

máximo absoluto en todo el archivo: 0.7638 → es el bloque circular del estudiante, y solo ese
máximo |r| en los bloques NO circulares: 0.0592 → ninguna variable que vaya a entrar al modelo se asocia con el objetivo


,bloque,|r| medio,|r| máximo
4,Directivo · DDIT41,0.017,0.036
5,Contexto escolar,0.017,0.030
2,Docente · PPIT45,0.035,0.035
3,Docente · PPIT46 (calidad de sus clases),0.040,0.056
1,Docente · PPIT44 (preparación),0.041,0.059
0,Estudiante · E3IT21,0.732,0.764


## 8. Preprocesamiento

Una sola función, `curar()`, con un log de decisiones. Cada paso dice qué hizo y
por qué; el log se imprime al final para que quede auditable.

| # | Paso | Decisión |
|---|---|---|
| 1 | Tipado | coordenadas y edad a numérico |
| 2 | Texto | normaliza acentos y espacios de la localidad |
| 3 | Coordenadas | una sola coordenada por escuela (mediana) |
| 4 | Duplicados | **estrategia B**: media de los dos docentes, primer valor del resto |
| 5 | `edad` | `edad_anos = edad + 6` |
| 6 | Índices | los dos que venían en 0–1 se multiplican por 100 |
| 7 | Asimetría | `log_num_hab_esc = log1p(num_hab_esc)` (683 a 45.769 habitantes) |
| 8 | Exclusión circular | fuera `E3IT21_*`, `Indice_preg`, `Indice_refuerzo` |
| 9 | Categóricas | `provincia` (13), `género` (2), `area_esc` (2) |

Las variables de baja varianza **no se borran**: los ítems del directivo y
`PPIT46` tienen 94–98 % de respuestas iguales, lo cual es un dato del instrumento,
no un error. Se marcan y se deja que los selectores decidan.

In [18]:
INDICES_0_1 = ["Indice_prep_didact_prof", "Indice_calidad_clases_prof"]
IDENTIFICADORES = ["IDSCHOOL", "IDCLASS", "IDSTUD", "Id_prof", "school", "nomb_est",
                   "nomb_esc", "test", "cod1_ubi_cen", "n_docentes"]
EXCLUIDAS_CIRCULARES = E3 + ["Indice_preg", "Indice_refuerzo", "Indice_global"]

log = []


def curar(datos):
    d = datos.copy()
    n_inicial, c_inicial = d.shape

    d["geo_esc_x"] = pd.to_numeric(d["geo_esc_x"], errors="coerce")
    d["geo_esc_y"] = pd.to_numeric(d["geo_esc_y"], errors="coerce")
    d["edad"] = pd.to_numeric(d["edad"], errors="coerce").astype("Int64")
    log.append(("1 · Tipado", "coordenadas y edad a numérico", ""))

    antes = d["ubi3_cen_esc"].nunique()
    d["ubi3_cen_esc"] = d["ubi3_cen_esc"].map(sin_acentos).str.strip()
    log.append(("2 · Texto", "localidad normalizada sin acentos/espacios",
                f"{antes} → {d.ubi3_cen_esc.nunique()} categorías"))

    # La provincia es el único nivel de geografía con cardinalidad manejable.
    d["provincia"] = d["ubi1_cen_esc"].map(sin_acentos).str.strip()

    coord = d.groupby("IDSCHOOL")[["geo_esc_x", "geo_esc_y"]].median()
    d = d.drop(columns=["geo_esc_x", "geo_esc_y"]).join(coord, on="IDSCHOOL")
    log.append(("3 · Coordenadas", "mediana por escuela (1 solo valor por escuela)",
                f"{(incoherentes > 1).any(axis=1).sum()} escuela(s) con 2 valores"))

    n_docentes = d.groupby("IDSTUD")["Id_prof"].nunique()
    medias = {c: "mean" for c in ITEM_PROF + IDX_PROF}
    primeras = {c: "first" for c in d.columns
                if c not in medias and c not in ("IDSTUD", "Id_prof")}
    d = d.groupby("IDSTUD", as_index=False).agg({**medias, **primeras})
    d["n_docentes"] = d["IDSTUD"].map(n_docentes).astype(int)
    log.append(("4 · Duplicados", "media de los 2 docentes (estrategia B)",
                f"{n_inicial} → {len(d)} filas"))

    d["edad_anos"] = d["edad"] + 6
    log.append(("5 · edad", "decodificada: edad_anos = edad + 6",
                f"rango {d.edad_anos.min()}–{d.edad_anos.max()} años"))

    for c in INDICES_0_1:
        d[c + "_100"] = d[c] * 100
    log.append(("6 · Índices", "los 2 índices en 0–1 multiplicados por 100",
                f"media {d.Indice_prep_didact_prof.mean():.2f} → "
                f"{d.Indice_prep_didact_prof_100.mean():.1f}"))

    d["log_num_hab_esc"] = np.log1p(d["num_hab_esc"])
    log.append(("7 · Asimetría", "log1p(num_hab_esc)",
                f"DE {d.num_hab_esc.std():,.0f} → {d.log_num_hab_esc.std():.2f}"))

    log.append(("8 · Exclusión circular",
                "se excluyen al definir FEATURES: " + ", ".join(EXCLUIDAS_CIRCULARES),
                "los componentes del objetivo no pueden ser variables del modelo"))

    log.append(("9 · Categóricas", "se conservan provincia (13), género (2), area_esc (2)",
                "municipio y localidad descartados (94 y 199 categorías)"))
    return d, pd.DataFrame(log, columns=["paso", "decisión", "efecto"])


curado, bitacora = curar(raw)
bitacora

,paso,decisión,efecto
0,1 · Tipado,coordenadas y edad a numérico,
1,2 · Texto,localidad normalizada sin acentos/espacios,199 → 192 categorías
2,3 · Coordenadas,mediana por escuela (1 solo valor por escuela),1 escuela(s) con 2 valores
3,4 · Duplicados,media de los 2 docentes (estrategia B),5595 → 4899 filas
4,5 · edad,decodificada: edad_anos = edad + 6,rango 7–14 años
5,6 · Índices,los 2 índices en 0–1 multiplicados por 100,media 0.51 → 50.8
6,7 · Asimetría,log1p(num_hab_esc),"DE 8,417 → 0.71"
7,8 · Exclusión circular,"se excluyen al definir FEATURES: E3IT21_01_N, ...",los componentes del objetivo no pueden ser var...
8,9 · Categóricas,"se conservan provincia (13), género (2), area_...",municipio y localidad descartados (94 y 199 ca...


In [19]:
FEATURES_NUM = (PPIT44 + PPIT45 + PPIT46 + DDIT41
                + ["Indice_prep_didact_prof_100", "Indice_calidad_clases_prof_100",
                   "Indice_calidad_clases_dir"]
                + ["edad_anos", "log_num_hab_esc", "num_hab_esc",
                   "can_prof_3", "prof_hom_3", "prof_muj_3"])
FEATURES_CAT = ["provincia", "género", "area_esc"]
FEATURES = FEATURES_NUM + FEATURES_CAT

assert not (set(FEATURES) & set(EXCLUIDAS_CIRCULARES)), "fuga: componente del objetivo"
assert not (set(FEATURES) & set(IDENTIFICADORES)), "un identificador no es una variable"
assert curado[FEATURES].notna().all().all(), "quedan nulos en el dataset curado"

print(f"estudiantes: {len(curado)} | variables: {len(FEATURES)} "
      f"({len(FEATURES_NUM)} numéricas + {len(FEATURES_CAT)} categóricas)")
print("\nAdvertencia de baja varianza (informativa, no se eliminan):")
varianza = (pd.DataFrame({"variable": FEATURES_NUM})
            .assign(share_modal=[curado[c].value_counts(normalize=True).max() for c in FEATURES_NUM],
                    n_valores=[curado[c].nunique() for c in FEATURES_NUM])
            .sort_values("share_modal", ascending=False))
print(varianza.head(8).to_string(index=False))

estudiantes: 4899 | variables: 28 (25 numéricas + 3 categóricas)

Advertencia de baja varianza (informativa, no se eliminan):
   variable  share_modal  n_valores
DDIT41_02_N        0.981          2
PPIT46_05_N        0.975          2
DDIT41_04_N        0.970          2
PPIT46_01_N        0.969          2
DDIT41_05_N        0.954          2
PPIT46_04_N        0.949          2
PPIT46_02_N        0.949          3
DDIT41_01_N        0.937          2


## 9. Los tres objetivos

Como el archivo no trae puntaje, el objetivo se toma de él mismo: el **índice
global de percepción del estudiante** (`Indice_global`, 0–100, media 48.6). Se
define en tres versiones para que el cuaderno de modelos ejercite los tres casos
de la regla de balanceo por IR.

| Tarea | Tipo | Objetivo | n | IR | Acción del proyecto |
|---|---|---|---|---|---|
| `reg_indice_global` | regresión | `Indice_global` (0–100) | 4.899 | — | — |
| `clf_indice_bajo` | clasificación | 0–40 (`bajo=1`) vs 60–100 | 4.899 | ≈1.05 | sin balancear |
| `clf_indice_nivel` | clasificación | 3 bandas (bajo/medio/alto) | 4.899 | ≈3.45 | SMOTE |

In [20]:
curado["bajo"] = (curado["Indice_global"] <= 40).astype(int)
curado["nivel"] = pd.cut(curado["Indice_global"], [-0.01, 40, 70, 100],
                         labels=["bajo", "medio", "alto"]).astype(str)

OBJETIVOS = {
    "reg_indice_global": {"tipo": "regresion", "columna": "Indice_global",
                          "descripcion": "Índice global de percepción del estudiante (0–100)"},
    "clf_indice_bajo": {"tipo": "clasificacion", "columna": "bajo",
                        "descripcion": "Percepción baja (0–40) vs alta (60–100)"},
    "clf_indice_nivel": {"tipo": "clasificacion", "columna": "nivel",
                         "descripcion": "Banda de percepción: bajo / medio / alto"},
}

resumen_obj = []
for tarea, spec in OBJETIVOS.items():
    y = curado[spec["columna"]]
    if spec["tipo"] == "regresion":
        ir = np.nan
        accion = "— (no aplica)"
        clases = {f"{y.min():.0f}–{y.max():.0f}": len(y)}
    else:
        conteo = y.value_counts()
        ir = float(BAL.calcular_ir(y.to_numpy()))
        accion = BAL.elegir_balanceo(ir, multiclase=y.nunique() > 2) or "sin balancear"
        clases = dict(conteo)
    resumen_obj.append({"tarea": tarea, "tipo": spec["tipo"], "objetivo": spec["columna"],
                        "media": round(float(y.mean()), 2) if spec["tipo"] == "regresion" else "—",
                        "clases / rango": str(clases),
                        "IR": round(ir, 2) if ir == ir else "—", "balanceo": accion})

resumen_obj = pd.DataFrame(resumen_obj)
print(f"media Indice_global = {curado.Indice_global.mean():.1f}  "
      f"(DE {curado.Indice_global.std():.1f}, rango {curado.Indice_global.min():.0f}–"
      f"{curado.Indice_global.max():.0f})\n")
resumen_obj

media Indice_global = 48.6  (DE 36.0, rango 0–100)



,tarea,tipo,objetivo,media,clases / rango,IR,balanceo
0,reg_indice_global,regresion,Indice_global,48.610,{'0–100': 4899},—,— (no aplica)
1,clf_indice_bajo,clasificacion,bajo,—,"{1: np.int64(2513), 0: np.int64(2386)}",1.050,sin balancear
2,clf_indice_nivel,clasificacion,nivel,—,"{'bajo': np.int64(2513), 'alto': np.int64(1658...",3.450,smote


In [21]:
fig = go.Figure()
fig.add_histogram(x=curado["Indice_global"], nbinsx=25, marker_color="#1f3a93",
                  opacity=0.85, name="Indice_global",
                  hovertemplate="Indice_global = %{x}<br>estudiantes: %{y}<extra></extra>")
fig.update_layout(
    title={"text": "Índice global de percepción del estudiante · 4.899 estudiantes de 3er grado",
           "x": 0.5, "xanchor": "center", "font": {"size": 17}},
    height=400, template="plotly_white", margin={"l": 60, "r": 20, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 13},
    xaxis={"title": "Indice_global (0–100)", "dtick": 20, "gridcolor": "#e8e8e8"},
    yaxis={"title": "estudiantes", "gridcolor": "#e8e8e8"},
    showlegend=False)
fig.show()

## 10. Selección de características con la metodología del proyecto

Se usan los cuatro evaluadores de `modelos/seleccion_atributos.py`, los mismos que
usa el bloque de modelos, y **dentro de una CV de 5 folds**: ajustar el selector
sobre todo el dataset y luego puntuar sobre el mismo dataset daría una selección
optimista. Cada fold vuelve a ajustar el preprocesador y el selector, y se cuenta
cuántas veces elige cada variable cada evaluador.

| Weka | Implementación | Búsqueda |
|---|---|---|
| `CfsSubsetEval` | mérito `k·mean(r_cf)/√(k+k(k−1)mean(r_ff))` | GreedyStepwise adelante |
| `ChiSquaredAttributeEval` | χ² (clasificación) / F (regresión) | Ranker top-8 |
| `InfoGainAttributeEval` | `mutual_info_*` | Ranker top-8 |
| `WrapperSubsetEval` | propio estimador (Ridge / LogReg) | GreedyStepwise, ≤8 |

Se lee la **frecuencia de selección** (estable = se elige en muchos folds) y el
**consenso** (elegida por varios evaluadores a la vez), no solo el subconjunto
ganador de un fold.

In [22]:
def seleccion_en_cv(df, tarea, spec, n_splits=mcfg.CV_FOLDS_SELECCION):
    # Ajusta los 4 evaluadores dentro de una CV y registra qué eligen.
    tipo = spec["tipo"]
    X_df = df[FEATURES]
    y = df[spec["columna"]].to_numpy()
    cortes = (StratifiedKFold(n_splits, shuffle=True, random_state=mcfg.SEED)
              if tipo == "clasificacion"
              else KFold(n_splits, shuffle=True, random_state=mcfg.SEED))

    registros = []
    for fold, (tr, va) in enumerate(cortes.split(X_df, y)):
        pre = MPP.construir_preprocesador(numericos=FEATURES_NUM, categoricos=FEATURES_CAT)
        X_tr = pre.fit_transform(X_df.iloc[tr])
        nombres = MPP.nombres_transformados(pre)
        for evaluador in mcfg.SELECTORES_NOMBRES:
            sel = SA.crear_selector(evaluador, tipo, seed=mcfg.SEED)
            sel.fit(X_tr, y[tr])
            elegidas = np.flatnonzero(np.asarray(sel.support_, dtype=bool))
            for i in elegidas:
                registros.append({"tarea": tarea, "fold": fold, "evaluador": evaluador,
                                  "variable": nombres[i], "tamaño": int(sel.support_.sum())})
    return pd.DataFrame(registros)


seleccion = pd.concat(
    [seleccion_en_cv(curado, tarea, spec) for tarea, spec in OBJETIVOS.items()],
    ignore_index=True)

print("selecciones registradas (folds × evaluadores × variables):", len(seleccion))
seleccion.groupby(["evaluador", "tarea"])["variable"].nunique().unstack()

selecciones registradas (folds × evaluadores × variables): 477


tarea,clf_indice_bajo,clf_indice_nivel,reg_indice_global
evaluador,,,
cfs,12,16,15
chi2,8,8,15
infogain,25,27,23
wrapper,7,8,14


## 11. Qué sobrevive a la selección

Tres lecturas:

- **Si una variable la elige un solo evaluador en un solo fold, no es señal**: sale
  del ruido del fold, no de un patrón.
- **La selección es inestable por construcción**: el objetivo es casi ortogonal a
  todo (§7), así que los evaluadores se ven forzados a mirar estas variables y el
  subconjunto cambia de fold a fold. Eso es información, no un defecto del método.
- Lo que se busca aquí no es "ganar" un subconjunto, sino **saber qué se puede
  afirmar** sobre estas variables antes de modelar.

In [23]:
freq = (seleccion.groupby(["tarea", "variable"])
        .agg(folds=("fold", "nunique"), evaluadores=("evaluador", "nunique"))
        .reset_index())
freq["frecuencia"] = (freq["folds"] / mcfg.CV_FOLDS_SELECCION).round(2)

# Tabla completa: las 28 variables × 3 tareas, con 0 para las que no se eligen nunca.
completo = (pd.MultiIndex.from_product([list(OBJETIVOS), FEATURES],
                                       names=["tarea", "variable"]).to_frame(index=False)
            .merge(freq, on=["tarea", "variable"], how="left")
            .fillna({"folds": 0, "evaluadores": 0, "frecuencia": 0.0})
            .astype({"folds": int, "evaluadores": int}))

consenso = (completo.groupby("variable")
            .agg(folds_totales=("folds", "sum"),
                 evaluadores_medios=("evaluadores", "mean"),
                 tareas_elegidas=("folds", lambda s: int((s > 0).sum())))
            .sort_values(["evaluadores_medios", "folds_totales"], ascending=False)
            .reset_index())
consenso["evaluadores_medios"] = consenso["evaluadores_medios"].round(2)
consenso["consenso"] = np.where(consenso.evaluadores_medios >= 3, "fuerte",
                                np.where(consenso.evaluadores_medios >= 2, "moderado", "débil"))
print(f"variables nunca seleccionadas por ningún evaluador en ningún fold: "
      f"{int((consenso.folds_totales == 0).sum())} de {len(FEATURES)}")
print("evaluadores que eligen por variable (media sobre las 3 tareas):")
consenso.head(15)

variables nunca seleccionadas por ningún evaluador en ningún fold: 4 de 28
evaluadores que eligen por variable (media sobre las 3 tareas):


,variable,folds_totales,evaluadores_medios,tareas_elegidas,consenso
0,Indice_prep_didact_prof_100,10,2.670,3,moderado
1,Indice_calidad_clases_prof_100,14,2.330,3,moderado
2,DDIT41_01_N,12,2.330,3,moderado
3,log_num_hab_esc,15,1.670,3,débil
4,num_hab_esc,15,1.670,3,débil
5,DDIT41_05_N,11,1.670,3,débil
6,PPIT44_01_N,7,1.670,3,débil
7,PPIT44_05_N,7,1.670,3,débil
8,can_prof_3,6,1.670,3,débil
9,PPIT46_05_N,10,1.330,2,débil


In [24]:
mapa = (completo.pivot_table(index="variable", columns="tarea", values="folds")
         .reindex(FEATURES).fillna(0).astype(int))
orden = ["reg_indice_global", "clf_indice_bajo", "clf_indice_nivel"]

fig = go.Figure(go.Heatmap(
    z=mapa[orden].to_numpy(), x=[f"{c} ({t})" for c, t in zip(orden, mapa[orden].columns)],
    y=mapa.index, zmin=0, zmax=mcfg.CV_FOLDS_SELECCION,
    colorscale=[[0, "#fbfbfa"], [1, "#1f3a93"]],
    text=mapa[orden].to_numpy().astype(int), texttemplate="%{text}",
    colorbar={"title": f"folds de {mcfg.CV_FOLDS_SELECCION}"}))
fig.update_layout(
    title={"text": "Folds en los que cada variable es seleccionada (4 evaluadores Weka)",
           "x": 0.5, "xanchor": "center", "font": {"size": 17}},
    height=max(560, 16 * len(mapa)), template="plotly_white",
    margin={"l": 230, "r": 20, "t": 60, "b": 80},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 12},
    xaxis={"tickangle": -20}, yaxis={"autorange": "reversed"})
fig.show()

In [25]:
top = (completo.groupby("variable")
       .agg(folds=("folds", "sum"), evaluadores=("evaluadores", "mean"))
       .reset_index()
       .merge(consenso[["variable", "consenso"]], on="variable")
       .sort_values(["evaluadores", "folds"], ascending=False).head(15))

fig = go.Figure(go.Bar(
    x=top.folds, y=top.variable, orientation="h",
    marker_color=[{"fuerte": "#1f3a93", "moderado": "#4aa3df", "débil": "#8c8c8c"}[c]
                  for c in top.consenso],
    text=[f"{e:.1f} ev · {f:.0f}" for e, f in zip(top.evaluadores, top.folds)],
    textposition="outside",
    hovertemplate="%{y}<br>evaluadores que la eligen / folds: %{x}<extra></extra>"))
fig.update_layout(
    title={"text": "Variables más frecuentemente seleccionadas (evaluadores · folds)",
           "x": 0.5, "xanchor": "center", "font": {"size": 17}},
    height=470, template="plotly_white", margin={"l": 210, "r": 60, "t": 60, "b": 50},
    font={"family": "Segoe UI, Helvetica, Arial, sans-serif", "size": 13},
    xaxis={"title": f"folds de {mcfg.CV_FOLDS_SELECCION} en los que se selecciona",
           "gridcolor": "#e8e8e8"},
    yaxis={"autorange": "reversed"})
fig.show()
print("legend: azul = consenso fuerte, celeste = moderado, gris = débil")

legend: azul = consenso fuerte, celeste = moderado, gris = débil


## 12. Escritura de los datasets curados

Se escriben en `modelos/datos/` con la convención del proyecto: **todas las
variables primero y el objetivo en la última columna**. Son archivos regenerables
(están en `.gitignore`) y son la entrada de `cuadernos/estimacion.ipynb`.

| Archivo | Contenido |
|---|---|
| `modelos/datos/reg_indice_global.csv` | 4.899 × 29, objetivo `Indice_global` |
| `modelos/datos/clf_indice_bajo.csv` | 4.899 × 29, objetivo `bajo` (0/1) |
| `modelos/datos/clf_indice_nivel.csv` | 4.899 × 29, objetivo `nivel` (bajo/medio/alto) |
| `modelos/datos/seleccion_indice_global.csv` | informe de selección de §10-11 |

El nombre de la columna objetivo es el de la variable (`Indice_global`, `bajo`,
`nivel`), igual que en los datasets que genera `python -m modelos.datos`.

In [26]:
rutas = {}
for tarea, spec in OBJETIVOS.items():
    tabla = pd.concat([curado[FEATURES].reset_index(drop=True),
                        curado[spec["columna"]].reset_index(drop=True)], axis=1)
    tabla.columns = FEATURES + [spec["columna"]]
    ruta = os.path.join(mcfg.DATOS_DIR, f"{tarea}.csv")
    tabla.to_csv(ruta, index=False)
    rutas[tarea] = ruta

ruta_sel = os.path.join(mcfg.DATOS_DIR, "seleccion_indice_global.csv")
informe = completo.merge(consenso[["variable", "consenso", "evaluadores_medios"]],
                         on="variable", how="left")
informe.to_csv(ruta_sel, index=False)

pd.DataFrame([{"archivo": os.path.relpath(r, RAIZ),
               "filas": f"{len(pd.read_csv(r, usecols=[0])):,}".replace(",", "."),
               "columnas": f"{len(FEATURES)} + 1 ({OBJETIVOS[tarea]['columna']})"}
              for tarea, r in rutas.items()]
             + [{"archivo": os.path.relpath(ruta_sel, RAIZ),
                 "filas": str(len(informe)),
                 "columnas": "frecuencia de selección (28 variables × 3 tareas)"}])

,archivo,filas,columnas
0,modelos/datos/reg_indice_global.csv,4.899,28 + 1 (Indice_global)
1,modelos/datos/clf_indice_bajo.csv,4.899,28 + 1 (bajo)
2,modelos/datos/clf_indice_nivel.csv,4.899,28 + 1 (nivel)
3,modelos/datos/seleccion_indice_global.csv,84,frecuencia de selección (28 variables × 3 tareas)


In [27]:
# Contrato: se releen los archivos y se comprueba que sirven para modelar
for tarea, spec in OBJETIVOS.items():
    df = pd.read_csv(rutas[tarea])
    assert df.shape[1] == len(FEATURES) + 1, f"{tarea}: nº de columnas inesperado"
    assert df.columns[-1] == spec["columna"], f"{tarea}: el objetivo no es la última columna"
    assert df.iloc[:, :-1].notna().all().all(), f"{tarea}: hay nulos en las variables"
    assert not set(df.columns[:-1]) & set(EXCLUIDAS_CIRCULARES), f"{tarea}: fuga circular"
    assert df[spec["columna"]].notna().all(), f"{tarea}: objetivo con nulos"
    objetivo = df.iloc[:, -1]
    if spec["tipo"] == "clasificacion":
        dist = {str(k): int(v) for k, v in objetivo.value_counts().head(3).items()}
    else:
        dist = f"{objetivo.mean():.1f} ± {objetivo.std():.1f}"
    print(f"✓ {tarea:18s} {df.shape} · objetivo = {df.columns[-1]!r} · distribución: {dist}")

print("\nEstudiantes por sección en el dataset curado:",
      curado.groupby("IDCLASS").size().describe()[["min", "50%", "max"]].round(1).to_dict())

✓ reg_indice_global  (4899, 29) · objetivo = 'Indice_global' · distribución: 48.6 ± 36.0
✓ clf_indice_bajo    (4899, 29) · objetivo = 'bajo' · distribución: {'1': 2513, '0': 2386}
✓ clf_indice_nivel   (4899, 29) · objetivo = 'nivel' · distribución: {'bajo': 2513, 'alto': 1658, 'medio': 728}

Estudiantes por sección en el dataset curado: {'min': 1.0, '50%': 22.0, 'max': 52.0}


## 13. Hallazgos y pendientes

**Sobre el archivo**

1. Los **6 índices son medias exactas de sus propios ítems** (§6). Por eso ningún
   objetivo de este archivo puede usar sus componentes como variables: la
   regresión da R² = 1.0000 sin aprender nada.
2. Las tres fuentes del módulo son **ortogonales** (§7): |r| máximo con el objetivo
   = 0.06. Con un solo objetivo, este instrumento no permite estimar el
   desempeño; permite describir tres cosas que no se tocan.
3. El grano es **estudiante × docente**: 696 estudiantes con dos informes (§3).
4. `edad` es un **código de banda**, no años (§4.1). En `erce/data.py` entra como
   numérica sin decodificar; el orden se respeta, pero la magnitud no es una edad.
5. `Indice_prep_didact_prof` y `Indice_calidad_clases_prof` vienen en **0–1**
   mientras los otros cuatro en 0–100 (§4). Aquí se normalizan; el pipeline
   existente no lo hace.
6. `sec_3` es **duplicada** de `can_prof_3`, que a su vez es
   `prof_hom_3 + prof_muj_3`: tres columnas, dos grados de libertad.
7. `Indice_calidad_clases_dir` tiene **85 % de respuestas en 100**: casi no
   discrimina (se conserva marcada, no se borra).

**Qué hace el preprocesamiento de este cuaderno**

Duplicados promediados (no solo la primera fila), `edad` decodificada, índices a
0–100, `log1p` en habitantes, componentes del objetivo excluidos, 28 variables.

**Continuación:** `cuadernos/estimacion.ipynb` carga estos tres datasets y construye los
modelos con el proceso completo (holdout, CV de selección, Friedman/Nemenyi,
validación imparcial, CV 10-fold, sobreajuste y artefactos).